# ProtoStar — controlled XGBoost experiment

This notebook tests whether XGBoost adds value beyond the starter Random Forest and the average/trend blend. It uses only the curated pre-2019 data and does not write `submission.csv`.

**Experiment:** one pooled XGBoost model per metric, trained on leakage-safe historical features and evaluated with recursive forecasts.

## Evaluation design

Hyperparameters are fixed before looking at the final results. We first inspect performance on the development period (train through 2002, predict 2003–2010), then evaluate unchanged models on the same final historical period used by the earlier notebooks (train through 2010, predict 2011–2018).

Forecasts are recursive: after predicting one year, that prediction—not the actual value—is used to build features for the next year. This matches the real 2019–2025 task.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from xgboost import XGBRegressor

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "code" / "common.py").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Open this notebook inside the ProtoStar-Challenge repository.")

CODE_DIR = PROJECT_ROOT / "code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from common import CUTOFF_YEAR, METRICS, load_valid_annual, mae, thirty_year_mean, wide
from starter_model import backtest, baselines

annual = load_valid_annual()
cities = pd.read_csv(PROJECT_ROOT / "data" / "city_info.csv", dtype={"city_id": str}).set_index("city_id")
print(f"Loaded {len(annual):,} guarded city-years and {len(cities)} city records.")

## Leakage-safe features

Every feature for target year `t` uses values from `t-1` or earlier: three lags, rolling means and variability, a recent 10-year slope, a recent anomaly, and static latitude/longitude. XGBoost handles missing feature values directly.

In [ ]:
FEATURE_COLUMNS = [
    "year", "lat", "lon",
    "lag_1", "lag_2", "lag_3",
    "rolling_mean_5", "rolling_std_5",
    "rolling_mean_10", "rolling_mean_30",
    "trend_10", "recent_anomaly",
]

def finite_slope(values: np.ndarray, min_observations: int = 5) -> float:
    values = np.asarray(values, dtype=float)
    observed = np.isfinite(values)
    if observed.sum() < min_observations:
        return np.nan
    x = np.arange(len(values), dtype=float)[observed]
    return float(np.polyfit(x, values[observed], 1)[0])

def historical_feature_tables(table: pd.DataFrame) -> dict[str, pd.DataFrame]:
    past = table.shift(1, axis=1)
    rolling_mean_10 = past.T.rolling(10, min_periods=5).mean().T
    return {
        "lag_1": table.shift(1, axis=1),
        "lag_2": table.shift(2, axis=1),
        "lag_3": table.shift(3, axis=1),
        "rolling_mean_5": past.T.rolling(5, min_periods=3).mean().T,
        "rolling_std_5": past.T.rolling(5, min_periods=3).std().T,
        "rolling_mean_10": rolling_mean_10,
        "rolling_mean_30": past.T.rolling(30, min_periods=10).mean().T,
        "trend_10": past.T.rolling(10, min_periods=5).apply(finite_slope, raw=True).T,
        "recent_anomaly": table.shift(1, axis=1) - rolling_mean_10,
    }

def build_training_frame(table: pd.DataFrame, last_train_year: int) -> pd.DataFrame:
    history = table.loc[:, :last_train_year]
    features = historical_feature_tables(history)
    frame = pd.concat(
        {name: values.stack(future_stack=True) for name, values in features.items()}, axis=1
    )
    frame["target"] = history.stack(future_stack=True)
    frame = frame.reset_index()
    frame["lat"] = frame["city_id"].map(cities["lat"])
    frame["lon"] = frame["city_id"].map(cities["lon"])
    return frame.dropna(subset=["target"]).sort_values(["year", "city_id"])

sample_training = build_training_frame(wide(annual, "annual_avg_temp"), 2010)
print(f"Training example: {len(sample_training):,} rows × {len(FEATURE_COLUMNS)} features")
sample_training.tail()

## Fixed XGBoost model

These are deliberately conservative CPU settings. This dataset is small enough that a GPU is unnecessary. No parameter search is performed on the 2011–2018 evaluation period.

In [ ]:
XGB_PARAMS = {
    "objective": "reg:absoluteerror",
    "n_estimators": 300,
    "learning_rate": 0.03,
    "max_depth": 3,
    "min_child_weight": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 5.0,
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": 1,
}

def features_for_year(history: pd.DataFrame, year: int) -> pd.DataFrame:
    prior = history.loc[:, history.columns < year]
    frame = pd.DataFrame(index=history.index)
    frame["year"] = year
    frame["lat"] = frame.index.map(cities["lat"])
    frame["lon"] = frame.index.map(cities["lon"])
    for lag in (1, 2, 3):
        frame[f"lag_{lag}"] = history[year - lag] if year - lag in history else np.nan
    frame["rolling_mean_5"] = prior.iloc[:, -5:].mean(axis=1)
    frame["rolling_std_5"] = prior.iloc[:, -5:].std(axis=1)
    frame["rolling_mean_10"] = prior.iloc[:, -10:].mean(axis=1)
    frame["rolling_mean_30"] = prior.iloc[:, -30:].mean(axis=1)
    frame["trend_10"] = prior.iloc[:, -10:].apply(lambda row: finite_slope(row.to_numpy()), axis=1)
    frame["recent_anomaly"] = frame["lag_1"] - frame["rolling_mean_10"]
    return frame[FEATURE_COLUMNS]

def forecast_xgboost(table: pd.DataFrame, last_train_year: int, years: list[int]):
    training = build_training_frame(table, last_train_year)
    model = XGBRegressor(**XGB_PARAMS)
    model.fit(training[FEATURE_COLUMNS], training["target"], verbose=False)

    history = table.loc[:, :last_train_year].copy()
    predictions = {}
    for year in years:
        predictions[year] = pd.Series(
            model.predict(features_for_year(history, year)), index=table.index
        )
        history[year] = predictions[year]
    return pd.DataFrame(predictions), model

XGB_PARAMS

## Development-period check

This first run is descriptive rather than a hyperparameter search: train through 2002 and recursively predict 2003–2010.

In [ ]:
def evaluate_xgboost(last_train_year: int, years: list[int]):
    rows, predictions, models = [], {}, {}
    for metric in METRICS:
        table = wide(annual, metric)
        prediction, model = forecast_xgboost(table, last_train_year, years)
        actual = table[years].where(thirty_year_mean(table, last_train_year).notna(), axis=0)
        rows.append({
            "model": "xgboost_engineered",
            "metric": metric,
            "mae": mae(actual.stack(future_stack=True), prediction.stack(future_stack=True)),
        })
        predictions[metric], models[metric] = prediction, model
    return pd.DataFrame(rows), predictions, models

development_report, _, _ = evaluate_xgboost(2002, list(range(2003, 2011)))
development_report.round(4)

## Final historical evaluation

The fixed XGBoost setup now predicts 2011–2018 from training data through 2010. For context, the previously selected blend weights are reproduced exactly from notebook 03; they are not reselected here.

In [ ]:
FINAL_LAST_TRAIN_YEAR = 2010
FINAL_YEARS = list(range(2011, CUTOFF_YEAR + 1))
BLEND_TREND_WEIGHTS = {
    "annual_avg_temp": 0.50,
    "total_precip_in": 0.25,
    "hottest_day_temp": 0.50,
    "cdd": 0.75,
}

def fixed_blend_score(metric: str) -> float:
    table = wide(annual, metric)
    references = baselines(table, FINAL_LAST_TRAIN_YEAR, FINAL_YEARS)
    weight = BLEND_TREND_WEIGHTS[metric]
    prediction = (1 - weight) * references["30_year_average"] + weight * references["linear_trend"]
    actual = table[FINAL_YEARS].where(thirty_year_mean(table, FINAL_LAST_TRAIN_YEAR).notna(), axis=0)
    return mae(actual.stack(future_stack=True), prediction.stack(future_stack=True))

comparison = backtest(annual, last_year=FINAL_LAST_TRAIN_YEAR)
blend_rows = pd.DataFrame([
    {"model": "selected_average_trend_blend", "metric": metric, "mae": fixed_blend_score(metric)}
    for metric in METRICS
])
xgb_report, xgb_predictions, xgb_models = evaluate_xgboost(FINAL_LAST_TRAIN_YEAR, FINAL_YEARS)
comparison = pd.concat([comparison, blend_rows, xgb_report], ignore_index=True)

reference_mae = comparison.loc[comparison["model"].eq("30_year_average"), ["metric", "mae"]].set_index("metric")["mae"]
comparison["skill_vs_30_year_average"] = 1 - comparison["mae"] / comparison["metric"].map(reference_mae)
comparison.pivot(index="model", columns="metric", values="mae").round(3)

In [ ]:
scores = 100 * comparison.groupby("model")["skill_vs_30_year_average"].mean().clip(lower=0)
display(comparison.sort_values(["metric", "mae"]).round(4))
print("Overall challenge-style score:")
scores.sort_values(ascending=False).round(2)

## What did XGBoost use?

Feature importance is not causality, but it helps diagnose whether the model mostly relies on recent observations, longer summaries, geography, or the calendar year.

In [ ]:
importance = pd.DataFrame({
    metric: model.feature_importances_ for metric, model in xgb_models.items()
}, index=FEATURE_COLUMNS)
importance["mean_importance"] = importance.mean(axis=1)
importance = importance.sort_values("mean_importance", ascending=False)
display(importance.round(4))

ax = importance["mean_importance"].sort_values().plot(kind="barh", figsize=(8, 5))
ax.set_title("Mean XGBoost feature importance across four metrics")
ax.set_xlabel("Gain-based importance")
plt.tight_layout()

## Key insights and decision

- Engineered XGBoost scores **5.81** on the 2011–2018 evaluation. That beats the pure linear trend (`5.03`) and starter RF (`0`), but not the simpler average/trend blend (`6.47`). XGBoost should therefore **not replace the blend as the overall lead model**.
- XGBoost is strongest for CDD: `169.19 °F-days` MAE, better than the linear trend (`174.60`), blend (`178.81`), and starter RF (`197.35`). This suggests a possible metric-specific role rather than using XGBoost for all four targets.
- It is weaker than the best simple choice for the other metrics: `1.53 °F` for average temperature, `6.28 in` for precipitation, and `2.63 °F` for hottest day. The blend or a reference baseline remains preferable for those targets.
- Development-period CDD error (`223.37`) differs sharply from final-period CDD error (`169.19`), showing that XGBoost performance is sensitive to the historical period. Additional expanding-window folds are required before trusting the CDD gain.
- Feature importance is dominated by 5-year means, 10-year means, and lag 1. Explicit `trend_10` and `year` contribute very little, confirming that this tree model mainly smooths recent history rather than extrapolating a long-term trend.
- A T4 GPU is unnecessary here: roughly 23,000 training rows, 12 features, and 300 shallow trees run quickly on CPU. GPU acceleration would matter more for the 10-million-row daily data or a large parameter search.

## Next decision

Keep the average/trend blend as the current overall leader. Next, validate both methods across multiple expanding-window folds; if the CDD advantage remains stable, consider a metric-specific system that uses XGBoost for CDD and the simpler blend/reference choices elsewhere. Do not write forecast values manually—a promoted method must be implemented in `code/` and regenerate `submission.csv` reproducibly.